# **EDA Notebook**



---
## 0. Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 75.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [3]:
# <Student to fill this section>
group_name = "Neural Minds"
student_name = "Aryaman Malik"
student_id = "25980501"

In [4]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [5]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [6]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# <Student to fill this section>

### 0.b Import Packages

In [7]:
# <Student to fill this section>
import pandas as pd
import altair as alt

---
## B. Data Understanding

In [8]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "store.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [9]:
# <Student to fill this section>
# Inspect shape, schema and basic structure of the store dataset
print("Shape:", df.shape)
df.info()
df.head()

Shape: (701, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 701 entries, 0 to 700
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   store_id         701 non-null    object
 1   sales_person_id  701 non-null    object
 2   name             701 non-null    object
dtypes: object(3)
memory usage: 16.6+ KB


,store_id,sales_person_id,name
0,e88ca038-7d3c-4343-a8cf-a273d7162426,28141e19-49e8-4090-9196-6e8f43b10de8,Next-Door Bike Store
1,48cab4d0-b342-451e-8f51-c0deeba6cbc4,16eb6889-25f7-475a-9eff-92024a00fde9,Professional Sales and Service
2,8b7b3e1a-51b7-4f62-9633-190ad3fa4471,0ca60d7b-b278-4a10-88c7-aa2c62611482,Riders Company
3,f1de0f2a-8c21-49eb-9631-01fd08c7e49b,0e8f1395-12c4-4d5b-b2df-257bc78a9b29,The Bike Mechanics
4,562eb541-ed31-4a78-b58f-ec1ecf57dea7,9787b7d5-2461-464d-9354-2da1defdb675,Nationwide Supply


In [10]:
# Check duplicates, missing values and uniqueness
print("Missing values per column:")
print(df.isnull().sum())
print("\nTotal duplicate rows:", df.duplicated().sum())
print("Duplicate store_id values:", df['store_id'].duplicated().sum())
print("\nUnique counts:")
for c in df.columns:
    print(f"  {c}: {df[c].nunique()} unique values")

Missing values per column:
store_id           0
sales_person_id    0
name               0
dtype: int64

Total duplicate rows: 0
Duplicate store_id values: 0

Unique counts:
  store_id: 701 unique values
  sales_person_id: 13 unique values
  name: 699 unique values


In [11]:
# <Student to fill this section>
dataset_insights = """
The store dataset contains 701 rows and 3 columns: store_id, sales_person_id and name.
All three are textual (object) and there are no missing values or fully duplicated rows.
- store_id is a UUID and is unique across all 701 records, confirming it is the primary key.
- sales_person_id has only 13 distinct values, meaning each salesperson is assigned to many stores - this is a categorical foreign key linking the store to a salesperson (one-to-many).
- name has 699 unique values, with only 2 store names appearing twice ("Friendly Bike Shop" and "Sports Products Store"). These are not data quality errors because the store_id values differ - they are simply two distinct stores that share a brand-like name.

Implications for the anomaly-detection use case (unusual sales line items):
The store table is small and stable. It will be joined indirectly: a sales order is placed by a customer, the customer belongs to a store (B2B reseller), and the store is owned by a salesperson. The sales_person_id column is the most useful feature for downstream modelling
because it gives us a categorical grouping that can be used to detect "lines outside this salesperson's typical pattern" if needed.
"""

In [12]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `\<sales_person_id\>`

In [13]:
# <Student to fill this section>
# Distribution of stores per salesperson
sp_counts = (
    df.groupby('sales_person_id').size()
      .reset_index(name='store_count')
      .sort_values('store_count', ascending=False)
)
print("Stores per salesperson:")
print(sp_counts)
print(f"\nMin: {sp_counts['store_count'].min()}, "
      f"Max: {sp_counts['store_count'].max()}, "
      f"Mean: {sp_counts['store_count'].mean():.1f}, "
      f"Median: {sp_counts['store_count'].median()}, "
      f"Std: {sp_counts['store_count'].std():.1f}")

Stores per salesperson:
                         sales_person_id  store_count
5   28141e19-49e8-4090-9196-6e8f43b10de8           80
8   59948fbd-99d9-43fd-b5ab-20614fc4193d           79
2   0e8f1395-12c4-4d5b-b2df-257bc78a9b29           77
1   0ca60d7b-b278-4a10-88c7-aa2c62611482           76
4   27d62980-8d18-4170-95a5-7f7bb9cada5a           74
12  f077eda9-c3db-4fde-81d2-e0f7ada08fcc           40
0   0445ab08-f703-4d2c-8ab3-fb7948bc1a48           40
6   3c91dda3-5a66-476e-b9d4-b686e959bda5           40
9   9737ee67-ea97-4ab7-9e6c-70f8878c6ab2           40
10  9787b7d5-2461-464d-9354-2da1defdb675           40
3   16eb6889-25f7-475a-9eff-92024a00fde9           39
7   48505c49-f9c7-4476-a670-158f89c331a7           38
11  e03321bf-f1d3-4324-9184-008d6fb535da           38

Min: 38, Max: 80, Mean: 53.9, Median: 40.0, Std: 19.2


In [14]:
# Bar chart - stores per salesperson
chart = alt.Chart(sp_counts).mark_bar(color='#4C78A8').encode(
    x=alt.X('sales_person_id:N', sort='-y', title='Sales Person', axis=alt.Axis(labels=False)),
    y=alt.Y('store_count:Q', title='Number of Stores'),
    tooltip=['sales_person_id', 'store_count']
).properties(
    title='Number of Stores per Salesperson',
    width=600, height=300
)
chart

alt.Chart(...)

In [15]:
# <Student to fill this section>
feature_1_insights = """
sales_person_id is a categorical foreign key with only 13 distinct values. The distribution is clearly bimodal:
- 5 "top-tier" salespersons each manage 74-80 stores (collectively 386 stores, ~55% of the portfolio).
- 8 "regular" salespersons each manage 38-40 stores (collectively 315 stores, ~45%).
The gap between the two tiers (40 vs 74) is large and not gradual, suggesting an internal segmentation rule rather than organic variation (for example, senior account managers vs junior ones, or regional vs national reps).

Limitations / issues:
- No date column is available, so we cannot tell if the assignment changed over time.
- 13 categories will need to be either one-hot encoded or target-encoded if this feature is used in the modelling notebook; one-hot is acceptable given the low cardinality.
- The store table itself contains no transactional information, so this feature on its own cannot drive anomaly detection - it must be joined to sales_order_header via customer.csv (customer -> store_id, header -> customer_id).
"""

In [16]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `\<name\>`

In [17]:
# <Student to fill this section>
# Investigate the store name column - duplicates, length, common starters
df['name_len'] = df['name'].str.len()
print("Name length statistics:")
print(df['name_len'].describe())

print("\nDuplicate store names (different store_id, same name):")
print(df[df['name'].duplicated(keep=False)].sort_values('name'))

print("\nTop 10 first words in store names:")
print(df['name'].str.split().str[0].value_counts().head(10))

Name length statistics:
count    701.000000
mean      20.124108
std        5.437988
min        9.000000
25%       16.000000
50%       20.000000
75%       24.000000
max       41.000000
Name: name_len, dtype: float64

Duplicate store names (different store_id, same name):
                                 store_id  \
235  66f0f15a-383f-4093-a0ba-2e73925df9e1   
540  13b206f0-9b50-4894-96be-28a79f271974   
542  45d9e9fd-131b-4671-bc11-f5abfdf653f5   
558  bdea515e-0f4a-4bb4-a49d-d3460ef86cc8   

                          sales_person_id                   name  name_len  
235  e03321bf-f1d3-4324-9184-008d6fb535da     Friendly Bike Shop        18  
540  0e8f1395-12c4-4d5b-b2df-257bc78a9b29     Friendly Bike Shop        18  
542  16eb6889-25f7-475a-9eff-92024a00fde9  Sports Products Store        21  
558  3c91dda3-5a66-476e-b9d4-b686e959bda5  Sports Products Store        21  

Top 10 first words in store names:
name
The        10
Fitness     9
Bike        9
Racing      8
Retail      7
General

In [18]:
# Distribution of store name lengths
hist = alt.Chart(df).mark_bar(color='#54A24B').encode(
    x=alt.X('name_len:Q', bin=alt.Bin(maxbins=20), title='Store name length (characters)'),
    y=alt.Y('count():Q', title='Number of stores'),
    tooltip=['count():Q']
).properties(
    title='Distribution of Store Name Lengths',
    width=600, height=300
)
hist

alt.Chart(...)

In [19]:
# <Student to fill this section>
feature_2_insights = """
The name column is high-cardinality text (699 unique values across 701 rows). Lengths range from 9 to 41 characters, mean ~20 (std ~5.4) - reasonable for English business names.
Only two name collisions exist - "Friendly Bike Shop" and "Sports Products Store" each appear twice with different store_id. These are valid records (same brand, two locations), not duplicates.
The top first words ("The", "Bike", "Fitness", "Racing", "Retail") indicate the stores are dominated by cycling and sporting goods resellers, which is consistent with this dataset being a B2B bicycle wholesaler (AdventureWorks).

Limitations / issues:
- Free-text store names are unlikely to be useful as a direct feature for the anomaly-detection model and would need to be either dropped or aggregated into a coarse category (for example, "Bike" vs "Fitness" vs "Other"). For the current use case (unusual sales line items) we will not engineer features from this column.
- The two duplicate names are worth flagging in a data-quality report but do not require fixing.
"""

In [20]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `\<store_id` (linkage to customer table)`\>`

In [21]:
# <Student to fill this section>
# How well does the store table connect to customer.csv?
customer_df = pd.read_csv(at.folder_path / "customer.csv")
print("Customers total:", len(customer_df))
print("Customers with a store_id:", customer_df['store_id'].notna().sum())
print("Customers without a store_id (individual / online):", customer_df['store_id'].isna().sum())

store_ids_in_customer = set(customer_df['store_id'].dropna().unique())
store_ids_in_store = set(df['store_id'].unique())
print(f"\nUnique store_ids in customer.csv: {len(store_ids_in_customer)}")
print(f"Unique store_ids in store.csv:    {len(store_ids_in_store)}")
print(f"Stores referenced by ≥1 customer: {len(store_ids_in_customer & store_ids_in_store)}")
print(f"Stores with NO customer:          {len(store_ids_in_store - store_ids_in_customer)}")
print(f"Customer store_ids missing from store table: {len(store_ids_in_customer - store_ids_in_store)}")

Customers total: 14275
Customers with a store_id: 452
Customers without a store_id (individual / online): 13823

Unique store_ids in customer.csv: 160
Unique store_ids in store.csv:    701
Stores referenced by ≥1 customer: 160
Stores with NO customer:          541
Customer store_ids missing from store table: 0


In [22]:
# Visualise customer / no-customer split among stores
coverage_df = pd.DataFrame({
    'segment': ['Stores with ≥1 customer', 'Stores with NO customer'],
    'count':   [160, 541]
})
chart = alt.Chart(coverage_df).mark_bar().encode(
    x=alt.X('segment:N', title=None),
    y=alt.Y('count:Q', title='Number of stores'),
    color=alt.Color('segment:N', legend=None,
                    scale=alt.Scale(range=['#4C78A8', '#F58518'])),
    tooltip=['segment', 'count']
).properties(
    title='Store usage in customer.csv',
    width=400, height=300
)
chart

alt.Chart(...)

In [23]:
# <Student to fill this section>
feature_n_insights = """
Linking store -> customer reveals a major data-coverage issue that directly affects the anomaly-detection use case:
- Only 160 of the 701 stores in store.csv are referenced by at least one customer. 541 stores (77%) are "orphan" - they exist in the master table but never appear as a customer's store, so they generate zero sales lines.
- Of the 14,275 customers, only 452 (3.2%) are B2B store customers; the remaining 13,823 (96.8%) are individual / online buyers with no store_id.
- All store_ids that DO appear in customer.csv are present in store.csv, so referential integrity in that direction is intact.

Implications for modelling:
1. The store dimension will only enrich about 452 customers and a small minority of sales lines. We should therefore use store-derived features as optional context, not as primary signal.
2. The high proportion of orphan stores suggests the master file was loaded from a CRM independently of the transactional system - typical of an SAP / AdventureWorks-style schema. This is not an error to fix but a fact to acknowledge.
3. For the anomaly-detection model we will primarily work at sales_order_detail level, with order_quantity, unit_price, discount, product cost/list price and special_offer as core features; salesperson and store context will only be appended for the subset of lines where it is available.
"""

In [24]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)